# VisualGuard-LVLM v3: audit, paired evaluation, full POPE (Kaggle)

What changed after your last run:
- **GPU memory crash fixed.** The notebook now picks the GPU with the most free memory, runs the decoder audit in a **separate process** (so its memory is released on exit), and avoids a 500 MB per-step allocation in the repo (same numerical result).
- **Resumable.** Every run saves its own JSON; finished runs are skipped. Full-split runs are done **one split at a time**, so a crash costs at most one split.
- **Reuses `baseline_full`** (your finished full-POPE baseline) as the comparison baseline whenever it exists.
- Each run writes into its **own folder** (the repo's VisualGuard variants otherwise overwrite each other's prediction files).

Order: 1 setup -> 2 data -> 3 audit -> 4 helpers -> 5 experiments -> 6 statistics -> 7 summary -> 8 optional CHAIR -> 9 bundle.
**Before running:** Session options -> GPU on, Internet on. For long runs use *Save Version -> Save & Run All*.

## 1. Setup

In [1]:
import glob, os, shutil, subprocess, zipfile, json, gc, time
from pathlib import Path

# ---- choose the GPU with the most free memory BEFORE torch is imported ----
def pick_gpu():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
        best = None
        for line in out.strip().splitlines():
            i, used, total = [int(x) for x in line.split(",")]
            if best is None or total - used > best[1]:
                best = (i, total - used)
        return best
    except Exception:
        return None

g = pick_gpu()
if g:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(g[0])
    print(f"using GPU {g[0]} ({g[1]} MiB free)")
    if g[1] < 12000:
        print("WARNING: <12 GB free. Restart the kernel (Run menu) to release GPU memory, then rerun this cell.")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

REPO = Path("/kaggle/working/VisualGuard-LVLM")
def first(*patterns):
    for pat in patterns:
        hits = sorted(glob.glob(pat))
        if hits:
            return hits[0]
    return None

if not REPO.exists():
    run_py = first(*[f"/kaggle/input/{'*/' * d}src/run.py" for d in range(9)])
    zp = first(*[f"/kaggle/input/{'*/' * d}VisualGuard*.zip" for d in range(9)])
    if run_py:
        shutil.copytree(Path(run_py).parent.parent, REPO)
    elif zp:
        with zipfile.ZipFile(zp) as z:
            z.extractall("/kaggle/working/_unz")
        shutil.copytree(next(Path("/kaggle/working/_unz").glob("*/src")).parent, REPO)
    else:
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/ria0304/VisualGuard-LVLM.git", str(REPO)], check=True)
os.chdir(REPO)
!pip install -q pycocotools sentencepiece pyarrow huggingface_hub accelerate "bitsandbytes>=0.43.0"

def patch(path, old, new, label):
    p = REPO / path; s = p.read_text()
    print(f"{label}:", "applied" if old in s else "not needed / already applied")
    p.write_text(s.replace(old, new))

# patch 1: Grounding DINO labels are a plain list in transformers 5.x
patch("src/model/grounding.py",
'''        for score, label, box in zip(
            results["scores"].tolist(),
            results.get("text_labels", results.get("labels")).tolist(),
            results["boxes"].tolist(),
        ):''',
'''        raw_labels = results.get("text_labels", results.get("labels"))
        if hasattr(raw_labels, "tolist"):
            raw_labels = raw_labels.tolist()
        for score, label, box in zip(
            results["scores"].tolist(),
            list(raw_labels),
            results["boxes"].tolist(),
        ):''', "grounding patch")

# patch 2: beam search crashes when length_penalty is None
patch("src/model/visual_guard_decoder.py",
'''                num_beams=self.decoding_config.num_beams,
                early_stopping=True,
            )''',
'''                num_beams=self.decoding_config.num_beams,
                early_stopping=True,
                length_penalty=(
                    1.0
                    if self.decoding_config.length_penalty is None
                    else self.decoding_config.length_penalty
                ),
            )''', "beam patch")

# patch 3: index the embedding table first, then cast (same values, avoids a ~500 MB allocation per step)
patch("src/model/visual_evidence.py",
"cand = F.normalize(token_embeddings.float()[list(candidate_token_ids)], dim=-1)",
"cand = F.normalize(token_embeddings[list(candidate_token_ids)].float(), dim=-1)", "memory patch")

# ---- settings ----
MODEL = "llava-hf/llava-1.5-7b-hf"
QUANT = "4bit"        # 7B fp16 does not fit a 15 GB GPU reliably; set None for the 0.5B model
SEED  = 42
N_AUDIT = 60          # samples for the decoder audit (Section 3)
N_ABL   = 300         # per-split samples for ablations (Section 5). Raise for tighter CIs
LAM_STAR = 5.0        # lambda for the 'effective' VisualGuard runs; set from the audit, then rerun this cell
QUANT_ARGS = f"--quantization {QUANT}" if QUANT else ""
DATA = Path("/kaggle/working/POPE"); IMG = DATA / "images"
OUT  = Path("/kaggle/working/results_v2"); OUT.mkdir(parents=True, exist_ok=True)   # same folder as before: keeps baseline_full
SETTINGS = ("random", "popular", "adversarial")
print("results folder:", OUT, "| existing runs:", sorted(p.name for p in OUT.iterdir() if p.is_dir()))

using GPU 0 (15360 MiB free)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.0 MB/s eta 0:00:00:00:0100:01
grounding patch: applied
beam patch: applied
memory patch: applied
results folder: /kaggle/working/results_v2 | existing runs: []


## 2. POPE data (all 3 splits; images come embedded in the HF parquet)

In [2]:
if not (DATA / "images").exists() or not any(DATA.glob("*.jsonl")):
    from huggingface_hub import hf_hub_download
    PQ = Path("/kaggle/working/pope_parquet"); PQ.mkdir(exist_ok=True)
    for s_ in SETTINGS:
        p_ = hf_hub_download("lmms-lab/POPE", f"Full/{s_}-00000-of-00001.parquet", repo_type="dataset")
        shutil.copy(p_, PQ / f"{s_}-00000-of-00001.parquet")
    !python output/convert_pope.py --parquet-dir {PQ} --data-root {DATA} --image-root {IMG}
for s_ in SETTINGS:
    f = next(DATA.glob(f"*{s_}*.jsonl"), None)
    print(s_, "->", f.name if f else "MISSING", "| rows:", sum(1 for _ in open(f)) if f else 0)

Full/random-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.0MB            

Full/random-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Full/popular-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.0MB            

Full/popular-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Full/adversarial-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.0MB            

Full/adversarial-00000-of-00001.parquet: downloading bytes:           |  0.00B            

random: wrote 3000 rows -> /kaggle/working/POPE/coco_pope_random.jsonl
popular: wrote 3000 rows -> /kaggle/working/POPE/coco_pope_popular.jsonl
adversarial: wrote 3000 rows -> /kaggle/working/POPE/coco_pope_adversarial.jsonl
images -> /kaggle/working/POPE/images
random -> coco_pope_random.jsonl | rows: 3000
popular -> coco_pope_popular.jsonl | rows: 3000
adversarial -> coco_pope_adversarial.jsonl | rows: 3000


## 3. Decoder audit (runs in its own process, so no GPU memory is left behind)
Penalty = `lam * max(0, threshold - VES) / threshold`, subtracted from the candidate logit, so it is **at most `lam` logits**. The script runs the real decoder once with `lam=1`, logs each top candidate's logit / evidence score / penalty for the first generated token, and we compute offline the **critical lambda**: the smallest lambda at which another candidate overtakes the top one. (`no` is in the repo's function-word list and gets zero penalty, so the penalty can only push `Yes` down.)

In [3]:
AUDIT_SRC = r'''
import argparse, json, sys
from pathlib import Path
ap = argparse.ArgumentParser()
ap.add_argument("--repo"); ap.add_argument("--model"); ap.add_argument("--quant", default="none")
ap.add_argument("--data"); ap.add_argument("--img"); ap.add_argument("--n", type=int, default=60); ap.add_argument("--out")
a = ap.parse_args()
sys.path.insert(0, a.repo)
from src.run import build_parser, build_experiment_configs
from src.model.visual_guard_decoder import VisualGuardDecoder
from src.evaluation.pope_eval import POPE_PROMPT_TEMPLATE, load_pope

argv = ["--benchmark", "pope", "--method", "visualguard", "--config", "visualguard", "--lambda", "1.0",
        "--model", a.model, "--device", "cuda", "--dtype", "float16", "--data-root", a.data]
if a.quant != "none": argv += ["--quantization", a.quant]
args = build_parser().parse_args(argv)
configs = build_experiment_configs(args)
print("evidence config:", configs["evidence"])
decoder = VisualGuardDecoder(lvlm_config=configs["lvlm"], decoding_config=configs["decoding"],
                             evidence_config=configs["evidence"], grounding_config=configs["grounding"], method=args.method)
decoder.load()
trace = []
_orig = decoder._select
def _traced(cands):
    chosen, did = _orig(cands)
    top = sorted(cands, key=lambda c: -c.logit)[:8]
    trace[-1]["steps"].append({"cands": [dict(text=c.text, logit=float(c.logit), ves=float(c.ves), penalty=float(c.penalty),
        content=bool(c.is_content), att=float(c.attention_evidence), sem=float(c.semantic_evidence)) for c in top],
        "chosen": chosen.text, "intervened": bool(did)})
    return chosen, did
decoder._select = _traced
per = max(1, a.n // 3)
for setting in ("random", "popular", "adversarial"):
    for smp in load_pope(pope_root=Path(a.data), setting=setting, image_root=Path(a.img), max_samples=per, require_images=True):
        trace.append({"setting": setting, "gt": smp.ground_truth, "steps": []})
        decoder.generate(smp.image_path, POPE_PROMPT_TEMPLATE.format(question=smp.question), max_new_tokens=4)
json.dump(trace, open(a.out, "w"))
print("traced", len(trace), "samples ->", a.out)
'''
Path("/kaggle/working/audit.py").write_text(AUDIT_SRC)
AUDIT_OUT = OUT / "audit_trace.json"
!python /kaggle/working/audit.py --repo {REPO} --model {MODEL} --quant {QUANT or 'none'} --data {DATA} --img {IMG} --n {N_AUDIT} --out {AUDIT_OUT} 2>&1 | grep -v "httpx\|HF_TOKEN\|Fetching\|Download\|Reconstruct\|Loading weights"
!nvidia-smi --query-gpu=index,memory.used --format=csv

evidence config: EvidenceConfig(alpha=1.0, beta=1.0, gamma=0.0, lam=1.0, threshold=0.35, layer_fraction=0.5, head_aggregation='mean', layer_aggregation='mean', attention_candidate_mix=0.5, normalize='minmax', clip_model_name='openai/clip-vit-base-patch32', clip_device=None, penalise_function_words=False)
traced 60 samples -> /kaggle/working/results_v2/audit_trace.json
index, memory.used [MiB]
0, 0 MiB
1, 0 MiB


In [4]:
import pandas as pd, numpy as np
trace = json.load(open(AUDIT_OUT))
rows = []
for t in trace:
    c = sorted(t["steps"][0]["cands"], key=lambda x: -x["logit"])
    top = c[0]; crit = None
    for o in c[1:]:
        dp = top["penalty"] - o["penalty"]
        if dp > 1e-9:
            v = (top["logit"] - o["logit"]) / dp
            crit = v if crit is None else min(crit, v)
    rows.append(dict(setting=t["setting"], top=top["text"], top_is_content=top["content"], top_penalty=top["penalty"],
                     top_ves=top["ves"], runner_up=c[1]["text"], runner_up_penalty=c[1]["penalty"],
                     logit_gap=top["logit"] - c[1]["logit"], critical_lambda=crit))
A = pd.DataFrame(rows)
print("first-token candidates at lam=1 (penalty = penalty per unit lambda)\n")
print(A.groupby("top").agg(n=("top", "size"), content=("top_is_content", "mean"), mean_pen=("top_penalty", "mean"),
                           mean_ves=("top_ves", "mean"), median_gap=("logit_gap", "median")).round(3))
print("\nlogit gap top1 vs top2:", A.logit_gap.describe().round(2).to_dict())
print("VES of top tokens     :", A.top_ves.describe().round(3).to_dict())
flip = A.critical_lambda.notna()
print(f"\nsamples where a penalty difference exists at all: {flip.sum()}/{len(A)}")
if flip.any():
    q = A.critical_lambda[flip].quantile([.25, .5, .75, .9]).round(2).tolist()
    print("critical lambda quantiles (25/50/75/90%):", q)
    print("suggested LAM_STAR (median critical lambda):", q[1])
for lam in (0.5, 1, 5, 20, 50):
    print(f"lambda {lam:>4}: answers that would flip = {int((A.critical_lambda <= lam).sum())}/{len(A)}")
display(A.head(10))

first-token candidates at lam=1 (penalty = penalty per unit lambda)

      n  content  mean_pen  mean_ves  median_gap
top                                             
No   31      0.0       0.0     0.493       3.219
Yes  29      1.0       0.0     0.502       2.234

logit gap top1 vs top2: {'count': 60.0, 'mean': 2.7, 'std': 1.17, 'min': 0.16, '25%': 1.84, '50%': 2.93, '75%': 3.64, 'max': 4.64}
VES of top tokens     : {'count': 60.0, 'mean': 0.497, 'std': 0.054, 'min': 0.419, '25%': 0.466, '50%': 0.496, '75%': 0.547, 'max': 0.591}

samples where a penalty difference exists at all: 0/60
lambda  0.5: answers that would flip = 0/60
lambda    1: answers that would flip = 0/60
lambda    5: answers that would flip = 0/60
lambda   20: answers that would flip = 0/60
lambda   50: answers that would flip = 0/60


,setting,top,top_is_content,top_penalty,top_ves,runner_up,runner_up_penalty,logit_gap,critical_lambda
0,random,Yes,True,0.0,0.427223,No,0.000000,1.531250,None
1,random,No,False,0.0,0.497583,Yes,0.013533,4.578125,None
2,random,Yes,True,0.0,0.427223,No,0.000000,2.234375,None
3,random,No,False,0.0,0.497583,Yes,0.013533,4.265625,None
4,random,Yes,True,0.0,0.418782,No,0.000000,0.953125,None
5,random,No,False,0.0,0.497583,Yes,0.013533,4.500000,None
6,random,Yes,True,0.0,0.591054,No,0.000000,3.000000,None
7,random,No,False,0.0,0.451116,Yes,0.000000,4.218750,None
8,random,Yes,True,0.0,0.573155,No,0.000000,2.296875,None
9,random,No,False,0.0,0.465661,Yes,0.000000,2.859375,None


**Reading Section 3.** If almost nothing can flip even at lambda 20-50, the penalty is too weak on one-word yes/no answers by design: that is a real finding to report. If `top_ves` barely varies, the evidence carries little signal for these tokens. To test a larger lambda, edit `LAM_STAR` in Section 1 and rerun that cell. **Caution:** choosing lambda on the same samples you report on is tuning on test data; Section 6 reports a held-out view for that reason.

## 4. Run helpers (own folder per run, skip finished runs, one split at a time for full runs)

In [5]:
GROUND = "--grounding-backend hf_grounding_dino"
FILTER = "grep -v 'httpx\\|HF_TOKEN\\|Fetching\\|Download\\|Reconstruct\\|Loading weights'"

def _done(out, stem): return (out / f"{stem}.json").exists()
def is_done(name):
    out = OUT / name
    return _done(out, name) or all(_done(out, f"{name}__{s}") for s in SETTINGS)

def run_cfg(name, method, extra="", n=N_ABL, grounding=False):
    # n=None -> full split, run one split per process so progress is saved after each split
    out = OUT / name; out.mkdir(parents=True, exist_ok=True)
    g = GROUND if grounding else ""
    jobs = [(None, name)] if n else [(s, f"{name}__{s}") for s in SETTINGS]
    for setting, stem in jobs:
        if _done(out, stem):
            print(f"[skip] {stem} already done"); continue
        maxs = f"--max-samples {n}" if n else ""
        sset = f"--setting {setting}" if setting else ""
        t0 = time.time()
        !python -m src.run --benchmark pope --method {method} {extra} --model {MODEL} --device cuda --dtype float16 \
            --data-root {DATA} --image-root {IMG} {maxs} {sset} --seed {SEED} {g} {QUANT_ARGS} \
            --output-dir {out} --run-name {stem} 2>&1 | grep -v "httpx\|HF_TOKEN\|Fetching\|Download\|Reconstruct\|Loading weights" | grep -v "^20.*\[.*\] [0-9]*/[0-9]* acc" 
        print(f"== {stem} finished in {(time.time()-t0)/60:.1f} min | done={_done(out, stem)}")

CFG = "--config visualguard"
# name -> (method, extra CLI args, needs grounding, description)
RUNS = {
  "vg_lamstar":     ("visualguard",    f"{CFG} --lambda {LAM_STAR}",             False, "attention+semantic, lambda=LAM_STAR"),
  "vg_default":     ("visualguard",    f"{CFG}",                                 False, "attention+semantic, yaml lambda 0.5"),
  "abl_attention":  ("attention",      f"{CFG} --lambda {LAM_STAR}",             False, "attention only"),
  "abl_semantic":   ("semantic",       f"{CFG} --lambda {LAM_STAR}",             False, "semantic only"),
  "abl_unidir":     ("unidirectional", f"{CFG} --lambda {LAM_STAR}",             False, "unidirectional variant"),
  "abl_region":     ("region",         f"{CFG} --lambda {LAM_STAR}",             True,  "region only"),
  "vg_full_region": ("visualguard",    f"{CFG} --lambda {LAM_STAR} --gamma 1.0", True,  "attention+semantic+region"),
  "sampling":       ("sampling",       "",                                       False, "established baseline"),
  "beam":           ("beam",           "",                                       False, "established baseline"),
}
# the comparison baseline: the full run if you have it, else a small run made here
BASE = "baseline_full" if is_done("baseline_full") else "baseline"
if BASE == "baseline":
    RUNS = {"baseline": ("baseline", "", False, "greedy decoding"), **RUNS}
print("comparison baseline:", BASE)
print(pd.DataFrame([(k, v[0], v[3]) for k, v in RUNS.items()], columns=["run", "method", "what"]).to_string(index=False))

comparison baseline: baseline
           run         method                                what
      baseline       baseline                     greedy decoding
    vg_lamstar    visualguard attention+semantic, lambda=LAM_STAR
    vg_default    visualguard attention+semantic, yaml lambda 0.5
 abl_attention      attention                      attention only
  abl_semantic       semantic                       semantic only
    abl_unidir unidirectional              unidirectional variant
    abl_region         region                         region only
vg_full_region    visualguard           attention+semantic+region
      sampling       sampling                established baseline
          beam           beam                established baseline


## 5. Experiments
**5a** smoke test (2 samples per split, tiny). **5b-1** the key question: does VisualGuard at `LAM_STAR` change anything? **5b-2** the remaining ablations. **5c** estimate full-run time, then run the full split for your chosen config.

In [6]:
SMOKE = OUT / "_smoke"
for name, (method, extra, g, _) in RUNS.items():
    out = SMOKE / name; out.mkdir(parents=True, exist_ok=True)
    gg = GROUND if g else ""
    print("--", name)
    !python -m src.run --benchmark pope --method {method} {extra} --model {MODEL} --device cuda --dtype float16 \
        --data-root {DATA} --image-root {IMG} --max-samples 2 --seed {SEED} {gg} {QUANT_ARGS} \
        --output-dir {out} --run-name {name} 2>&1 | grep -i "error\|Traceback\|Wrote results" | head -3
print("smoke test done: any 'Error'/'Traceback' above must be fixed before 5b")

-- baseline
2026-10-16 19:19:41,582 INFO    visualguard: Wrote results to /kaggle/working/results_v2/_smoke/baseline/baseline.json
-- vg_lamstar
2026-10-16 19:20:17,825 INFO    visualguard: Wrote results to /kaggle/working/results_v2/_smoke/vg_lamstar/vg_lamstar.json
-- vg_default
2026-10-16 19:20:54,486 INFO    visualguard: Wrote results to /kaggle/working/results_v2/_smoke/vg_default/vg_default.json
-- abl_attention
2026-10-16 19:21:22,414 INFO    visualguard: Wrote results to /kaggle/working/results_v2/_smoke/abl_attention/abl_attention.json
-- abl_semantic
2026-10-16 19:21:59,159 INFO    visualguard: Wrote results to /kaggle/working/results_v2/_smoke/abl_semantic/abl_semantic.json
-- abl_unidir
2026-10-16 19:22:35,616 INFO    visualguard: Wrote results to /kaggle/working/results_v2/_smoke/abl_unidir/abl_unidir.json
-- abl_region
2026-10-16 19:23:24,275 INFO    visualguard: Wrote results to /kaggle/working/results_v2/_smoke/abl_region/abl_region.json
-- vg_full_region
2026-10-16 19:

## 6. Paired statistics (defined before use)

In [7]:
from scipy.stats import binomtest
rng = np.random.default_rng(SEED)

def load_preds(run, method):
    out = {}
    for s_ in SETTINGS:
        f = OUT / run / f"pope_{s_}_{method}_predictions.jsonl"
        if f.exists():
            out[s_] = [json.loads(l) for l in open(f)]
    return out

def arrays(recs):
    y = np.array([str(r["ground_truth"]).lower() == "yes" for r in recs])
    p = np.array([str(r["prediction"]).lower() == "yes" for r in recs])
    lat = np.array([r.get("latency_s", np.nan) for r in recs], dtype=float)
    return y, p, lat

def metrics(y, p):
    tp = (y & p).sum(); fp = (~y & p).sum(); fn = (y & ~p).sum()
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    return dict(acc=(y == p).mean(), f1=2 * prec * rec / max(prec + rec, 1e-9), halluc=fp / max((~y).sum(), 1))

def boot_ci(y, p, key, B=2000):
    n = len(y); vals = []
    for _ in range(B):
        i = rng.integers(0, n, n); vals.append(metrics(y[i], p[i])[key])
    return np.percentile(vals, [2.5, 97.5])

def compare(base_run, base_method, run, method, skip_first=0):
    # pairs by question_id, so a 300/split run is compared with the same 300 questions of the full baseline
    bp, rp = load_preds(base_run, base_method), load_preds(run, method)
    rows = []
    for s_ in list(SETTINGS) + ["ALL"]:
        ss = SETTINGS if s_ == "ALL" else [s_]
        yb, pb, pr, lb, lr = [], [], [], [], []
        for k in ss:
            if k not in bp or k not in rp: continue
            R = {r["question_id"]: r for r in rp[k][skip_first:]}
            B = {r["question_id"]: r for r in bp[k]}
            common = [q for q in R if q in B]
            y, p0, l0 = arrays([B[q] for q in common]); _, p1, l1 = arrays([R[q] for q in common])
            yb.append(y); pb.append(p0); pr.append(p1); lb.append(l0); lr.append(l1)
        if not yb: continue
        y, p0, p1 = np.concatenate(yb), np.concatenate(pb), np.concatenate(pr)
        l0, l1 = np.concatenate(lb), np.concatenate(lr)
        c0, c1 = (y == p0), (y == p1)
        fixed, broke = int((~c0 & c1).sum()), int((c0 & ~c1).sum())
        pval = binomtest(min(fixed, broke), fixed + broke, 0.5).pvalue if fixed + broke else 1.0
        m0, m1 = metrics(y, p0), metrics(y, p1); lo, hi = boot_ci(y, p1, "acc")
        rows.append(dict(run=run, setting=s_, n=len(y), acc_base=m0["acc"], acc_run=m1["acc"], acc_ci=f"[{lo:.3f},{hi:.3f}]",
                         f1_base=m0["f1"], f1_run=m1["f1"], halluc_base=m0["halluc"], halluc_run=m1["halluc"],
                         changed=int((p0 != p1).sum()), fixed=fixed, broke=broke, mcnemar_p=pval,
                         time_x=np.nanmean(l1) / np.nanmean(l0)))
    return pd.DataFrame(rows)

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
def show_ablations():
    done = [r for r in RUNS if r != BASE and is_done(r)]
    if not done: print("no finished runs yet"); return
    ABL = pd.concat([compare(BASE, "baseline", r, RUNS[r][0]) for r in done], ignore_index=True)
    display(ABL[ABL.setting == "ALL"].round(4).drop(columns="setting"))
    ABL.to_csv(OUT / "ablation_paired.csv", index=False)
print("stats helpers ready")

stats helpers ready


In [8]:
# 5b-1 THE KEY CHECK: does VisualGuard at LAM_STAR change any answer?
if BASE == "baseline": run_cfg("baseline", "baseline", "", n=N_ABL)
run_cfg("vg_lamstar", *RUNS["vg_lamstar"][:2], n=N_ABL, grounding=RUNS["vg_lamstar"][2])
display(compare(BASE, "baseline", "vg_lamstar", "visualguard").round(4))

2026-10-16 19:25:20,138 INFO    visualguard: Method: baseline
2026-10-16 19:25:20,138 INFO    visualguard: LVLM: llava-hf/llava-1.5-7b-hf
2026-10-16 19:25:20,138 INFO    visualguard: Evidence weights: alpha=0.0 beta=0.0 gamma=0.0 lambda=0.0 threshold=1.0
2026-10-16 19:25:24,818 INFO    numexpr.utils: NumExpr defaulting to 4 threads.
2026-10-16 19:25:25,061 INFO    src.model.llava_backend: Loading LVLM llava-hf/llava-1.5-7b-hf (dtype=torch.float16, device=cuda, quant=4bit)
2026-10-16 19:25:36,898 INFO    src.evaluation.pope_eval: POPE baseline/random: 300 samples (images verified present)
2026-10-16 19:29:50,712 INFO    src.evaluation.pope_eval: Wrote 300 predictions to /kaggle/working/results_v2/baseline/pope_random_baseline_predictions.jsonl
2026-10-16 19:29:50,735 INFO    src.evaluation.pope_eval: POPE baseline/popular: 300 samples (images verified present)
2026-10-16 19:34:04,270 INFO    src.evaluation.pope_eval: Wrote 300 predictions to /kaggle/working/results_v2/baseline/pope_popu

,run,setting,n,acc_base,acc_run,acc_ci,f1_base,f1_run,halluc_base,halluc_run,changed,fixed,broke,mcnemar_p,time_x
0,vg_lamstar,random,300,0.8600,0.8533,"[0.810,0.890]",0.8444,0.8358,0.0400,0.0400,2,0,2,0.5000,2.1702
1,vg_lamstar,popular,300,0.8600,0.8533,"[0.810,0.893]",0.8444,0.8358,0.0400,0.0400,2,0,2,0.5000,2.1535
2,vg_lamstar,adversarial,300,0.8233,0.8167,"[0.773,0.860]",0.8114,0.8029,0.1133,0.1133,2,0,2,0.5000,2.1556
3,vg_lamstar,ALL,900,0.8478,0.8411,"[0.817,0.864]",0.8331,0.8245,0.0644,0.0644,6,0,6,0.0312,2.1597


**Read the table:** `changed` = answers that differ from the baseline; `fixed` = baseline wrong, VisualGuard right; `broke` = the reverse; `mcnemar_p` = chance of such a difference if both were equally good; `time_x` = slowdown. `changed = 0` means VisualGuard still does nothing at this lambda: do not start a full run.

In [9]:
# 5b-2 remaining ablations + established baselines (skips finished runs)
for name, (method, extra, g, _) in RUNS.items():
    if name == "vg_lamstar": continue
    run_cfg(name, method, extra, n=N_ABL, grounding=g)
show_ablations()

[skip] baseline already done
2026-10-16 20:06:05,393 INFO    visualguard: Method: visualguard
2026-10-16 20:06:05,393 INFO    visualguard: LVLM: llava-hf/llava-1.5-7b-hf
2026-10-16 20:06:05,393 INFO    visualguard: Evidence weights: alpha=1.0 beta=1.0 gamma=0.0 lambda=0.5 threshold=0.35
2026-10-16 20:06:10,061 INFO    numexpr.utils: NumExpr defaulting to 4 threads.
2026-10-16 20:06:10,286 INFO    src.model.llava_backend: Loading LVLM llava-hf/llava-1.5-7b-hf (dtype=torch.float16, device=cuda, quant=4bit)
2026-10-16 20:06:22,248 INFO    src.evaluation.pope_eval: POPE visualguard/random: 300 samples (images verified present)
2026-10-16 20:15:33,402 INFO    src.evaluation.pope_eval: Wrote 300 predictions to /kaggle/working/results_v2/vg_default/pope_random_visualguard_predictions.jsonl
2026-10-16 20:15:33,425 INFO    src.evaluation.pope_eval: POPE visualguard/popular: 300 samples (images verified present)
2026-10-16 20:24:40,537 INFO    src.evaluation.pope_eval: Wrote 300 predictions to /

,run,n,acc_base,acc_run,acc_ci,f1_base,f1_run,halluc_base,halluc_run,changed,fixed,broke,mcnemar_p,time_x
3,vg_lamstar,900,0.8478,0.8411,"[0.817,0.864]",0.8331,0.8245,0.0644,0.0644,6,0,6,0.0312,2.1597
7,vg_default,900,0.8478,0.8478,"[0.823,0.871]",0.8331,0.8331,0.0644,0.0644,0,0,0,1.0000,2.1615
11,abl_attention,900,0.8478,0.8478,"[0.824,0.871]",0.8331,0.8319,0.0644,0.0578,6,3,3,1.0000,1.2731
15,abl_semantic,900,0.8478,0.8322,"[0.807,0.857]",0.8331,0.8124,0.0644,0.0622,16,1,15,0.0005,2.1617
19,abl_unidir,900,0.8478,0.8411,"[0.818,0.866]",0.8331,0.8245,0.0644,0.0644,6,0,6,0.0312,2.1313
23,abl_region,900,0.8478,0.8278,"[0.802,0.852]",0.8331,0.8040,0.0644,0.0511,30,6,24,0.0014,2.1052
27,vg_full_region,900,0.8478,0.8478,"[0.823,0.871]",0.8331,0.8331,0.0644,0.0644,0,0,0,1.0000,3.0060
31,sampling,900,0.8478,0.8067,"[0.780,0.833]",0.8331,0.7943,0.0644,0.1333,99,31,68,0.0003,0.9977
35,beam,900,0.8478,0.8478,"[0.824,0.871]",0.8331,0.8331,0.0644,0.0644,0,0,0,1.0000,3.2297


In [10]:
# 5c time estimate for a FULL run, from the measured per-sample latency
lat = [r["latency_s"] for r in load_preds("vg_lamstar", "visualguard").get("random", []) if "latency_s" in r]
if lat:
    print(f"mean VisualGuard latency: {np.mean(lat):.2f} s/sample -> full POPE (9000 samples) ~ {np.mean(lat) * 9000 / 3600:.1f} h")
print("baseline_full took 148 min in your earlier run")

mean VisualGuard latency: 1.84 s/sample -> full POPE (9000 samples) ~ 4.6 h
baseline_full took 148 min in your earlier run


In [11]:
# 5c FULL POPE for your chosen config (split-by-split, resumable). Set BEST after reading the ablation table.
BEST = "vg_lamstar"
if not is_done("baseline_full"):
    run_cfg("baseline_full", "baseline", "", n=None)
b = RUNS[BEST]
run_cfg(f"{BEST}_full", b[0], b[1], n=None, grounding=b[2])

2026-10-16 23:45:26,681 INFO    visualguard: Method: baseline
2026-10-16 23:45:26,682 INFO    visualguard: LVLM: llava-hf/llava-1.5-7b-hf
2026-10-16 23:45:26,682 INFO    visualguard: Evidence weights: alpha=0.0 beta=0.0 gamma=0.0 lambda=0.0 threshold=1.0
2026-10-16 23:45:31,477 INFO    numexpr.utils: NumExpr defaulting to 4 threads.
2026-10-16 23:45:31,703 INFO    src.model.llava_backend: Loading LVLM llava-hf/llava-1.5-7b-hf (dtype=torch.float16, device=cuda, quant=4bit)
2026-10-16 23:45:43,763 INFO    src.evaluation.pope_eval: POPE baseline/random: 3000 samples (images verified present)
2026-10-17 00:27:51,214 INFO    src.evaluation.pope_eval: Wrote 3000 predictions to /kaggle/working/results_v2/baseline_full/pope_random_baseline_predictions.jsonl
2026-10-17 00:27:51,226 INFO    visualguard: Wrote results to /kaggle/working/results_v2/baseline_full/baseline_full__random.json

VisualGuard | benchmark=pope method=baseline model=llava-hf/llava-1.5-7b-hf
setting            n      acc    

## 7. Final tables: full POPE and a held-out view
Held-out = samples after the first `N_ABL` per split (the first `N_ABL` were used to choose `LAM_STAR` / `BEST`, so only the remainder is an unbiased estimate).

In [13]:
if is_done("baseline_full") and is_done(f"{BEST}_full"):
    meth = RUNS[BEST][0]
    FULL = compare("baseline_full", "baseline", f"{BEST}_full", meth)
    HELD = compare("baseline_full", "baseline", f"{BEST}_full", meth, skip_first=N_ABL)
    print("FULL POPE (3000/split)"); display(FULL.round(4))
    print(f"HELD-OUT (after the first {N_ABL} per split)"); display(HELD.round(4))
    FULL.to_csv(OUT / "final_full.csv", index=False); HELD.to_csv(OUT / "final_heldout.csv", index=False)
else:
    print("run the full cell in 5c first")
print()
for f in sorted(glob.glob(str(OUT / "*" / "*.json"))):
    if "_smoke" in f or "audit" in f: continue
    d = json.load(open(f))
    if "pope" in d:
        print(Path(f).stem.ljust(30), {k: round(v["metrics"]["f1"], 4) for k, v in d["pope"].items() if "f1" in v.get("metrics", {})},
              "| runtime_min:", round(d["efficiency"]["total_runtime_s"] / 60, 1))

run the full cell in 5c first

abl_attention                  {'adversarial': 0.8129, 'popular': 0.8433, 'random': 0.8401} | runtime_min: 16.4
abl_region                     {'adversarial': 0.7852, 'popular': 0.8154, 'random': 0.8123} | runtime_min: 27.0
abl_semantic                   {'adversarial': 0.7927, 'popular': 0.8226, 'random': 0.8226} | runtime_min: 27.7
abl_unidir                     {'adversarial': 0.8029, 'popular': 0.8358, 'random': 0.8358} | runtime_min: 27.3
baseline                       {'adversarial': 0.8114, 'popular': 0.8444, 'random': 0.8444} | runtime_min: 12.9
baseline_full__adversarial     {'adversarial': 0.8218} | runtime_min: 42.5
baseline_full__popular         {'popular': 0.8402} | runtime_min: 42.6
baseline_full__random          {'random': 0.8544} | runtime_min: 42.4
beam                           {'adversarial': 0.8114, 'popular': 0.8444, 'random': 0.8444} | runtime_min: 41.2
sampling                       {'adversarial': 0.7973, 'popular': 0.777, 'random'

**What counts as evidence of improvement:** `fixed` clearly above `broke`, a small McNemar p, a hallucination-rate drop whose confidence interval excludes zero on the **held-out** view, and acceptable `time_x`. If `changed` stays 0 even at a large lambda, the Section 3 audit explains why; that can be reported honestly as a negative result.

## 8. (Optional) CHAIR, with a preflight that counts missing images first

In [14]:
RUN_CHAIR = False   # set True only after the POPE sections are finished and saved
ANN  = first(*[f"/kaggle/input/{'*/' * d}annotations/instances_val2017.json" for d in range(9)])
CIMG = first(*[f"/kaggle/input/{'*/' * d}val2017" for d in range(9)])
print("annotations:", ANN); print("images dir :", CIMG)
missing = None
if ANN and CIMG:
    imgs = json.load(open(ANN))["images"]
    missing = [im["file_name"] for im in imgs if not (Path(CIMG) / im["file_name"]).is_file()]
    print(f"{len(imgs)} images in annotations | missing on disk: {len(missing)}", missing[:3])
else:
    print("COCO 2017 not found: add it via Add Input, or skip CHAIR")

annotations: /kaggle/input/datasets/sabahesaraki/2017-2017/annotations_trainval2017/annotations/instances_val2017.json
images dir : /kaggle/input/datasets/sabahesaraki/2017-2017/val2017
5000 images in annotations | missing on disk: 5000 ['000000397133.jpg', '000000037777.jpg', '000000252219.jpg']


In [15]:
if RUN_CHAIR and ANN and CIMG and missing == []:
    SYN = "/kaggle/working/synonyms.txt"
    !wget -q -O {SYN} https://raw.githubusercontent.com/LisaAnne/Hallucination/master/data/synonyms.txt
    ANN_DIR = str(Path(ANN).parent)
    for m, cfg in [("baseline", ""), ("visualguard", f"--config visualguard --lambda {LAM_STAR}")]:
        co = OUT / f"chair_{m}"; co.mkdir(exist_ok=True)
        !python scripts/run_chair.py --method {m} {cfg} --model {MODEL} --device cuda --dtype float16 {QUANT_ARGS} \
            --ann-dir {ANN_DIR} --image-root {CIMG} --coco-split val2017 --synonyms {SYN} \
            --num-images 100 --max-new-tokens 128 --seed {SEED} --output-dir {co}
else:
    print("CHAIR skipped (RUN_CHAIR=False, COCO missing, or images missing). val2017 also differs from the val2014 used in published CHAIR numbers.")

CHAIR skipped (RUN_CHAIR=False, COCO missing, or images missing). val2017 also differs from the val2014 used in published CHAIR numbers.


## 9. Bundle everything for download

In [16]:
shutil.make_archive("/kaggle/working/visualguard_v3_results", "zip", OUT)
print("Output panel -> /kaggle/working -> visualguard_v3_results.zip -> Download. Use Save Version to keep outputs after the session ends.")

Output panel -> /kaggle/working -> visualguard_v3_results.zip -> Download. Use Save Version to keep outputs after the session ends.
